In [110]:
import numpy as np
import pandas as pd


In [111]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [112]:
df = pd.read_csv('/Users/ankushkumar/AI-Ml/Ai-Ml-Lab/loan_approval.csv')


In [113]:
df.head()

,applicant_id,age,employment_type,city_tier,annual_income,loan_amount_lakh,credit_score,existing_loans,years_at_job,approved
0,APP1001,58,Salaried,Tier-2,736000,24.2,718.0,0,25.0,1
1,APP1002,46,Salaried,Tier-2,282000,5.6,712.0,4,6.0,0
2,APP1003,48,Salaried,Tier-2,931000,15.5,513.0,2,NaN,0
3,APP1004,56,Salaried,Tier-1,1300000,34.1,682.0,1,16.0,1
4,APP1005,44,Salaried,Tier-1,533000,7.5,742.0,0,12.0,1


In [114]:
# Q1.1 (Load) Load loan_approval.csv into df. Print its shape, then use .isnull().sum() to see which columns have missing values.

df.shape
df.info()
df.isnull().sum()


<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   applicant_id      500 non-null    str    
 1   age               500 non-null    int64  
 2   employment_type   500 non-null    str    
 3   city_tier         500 non-null    str    
 4   annual_income     500 non-null    int64  
 5   loan_amount_lakh  500 non-null    float64
 6   credit_score      480 non-null    float64
 7   existing_loans    500 non-null    int64  
 8   years_at_job      485 non-null    float64
 9   approved          500 non-null    int64  
dtypes: float64(3), int64(4), str(3)
memory usage: 50.1 KB


applicant_id         0
age                  0
employment_type      0
city_tier            0
annual_income        0
loan_amount_lakh     0
credit_score        20
existing_loans       0
years_at_job        15
approved             0
dtype: int64

In [115]:
# Q1.2 (Clean) Drop the applicant_id column, since it is only an identifier and carries no predictive signal. Fill the missing values in credit_score and years_at_job with the median of each column.

df.drop(columns=['applicant_id'], inplace=True)
df['credit_score'].fillna(df['credit_score'].median(), inplace=True)
df['years_at_job'].fillna(df['years_at_job'].median(), inplace=True)

/var/folders/26/8dmzs5m57bl3f9r887j35wk80000gn/T/ipykernel_63357/1305720671.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['credit_score'].fillna(df['credit_score'].median(), inplace=True)
/var/folders/26/8dmzs5m57bl3f9r887j35wk80000gn/T/ipykernel_63357/1305720671.py:5: ChainedAssignmentError: A value is being set on a copy o

0      25.0
1       6.0
2       8.0
3      16.0
4      12.0
       ... 
495    10.0
496     9.0
497    14.0
498    12.0
499     3.0
Name: years_at_job, Length: 500, dtype: float64

In [116]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               500 non-null    int64  
 1   employment_type   500 non-null    str    
 2   city_tier         500 non-null    str    
 3   annual_income     500 non-null    int64  
 4   loan_amount_lakh  500 non-null    float64
 5   credit_score      480 non-null    float64
 6   existing_loans    500 non-null    int64  
 7   years_at_job      485 non-null    float64
 8   approved          500 non-null    int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 42.8 KB


In [117]:
# Q1.3 (Split) Create X using every column except approved, and y using the approved column. Split them into X_train, X_test, y_train, y_test using test_size=0.2 and random_state=42.
x = df.drop(columns=['approved'])
y = df['approved']
print(X.shape, y.shape)


(500, 8) (500,)


In [118]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [119]:
# Q1.4 (Encode + Scale) Build a ColumnTransformer that applies OneHotEncoder to the text columns (employment_type, city_tier) and StandardScaler to the numeric columns (age, annual_income, loan_amount_lakh, credit_score, existing_loans, years_at_job).
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

transformer = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), ['employment_type', 'city_tier']),
        ('scaler', StandardScaler(), ['age', 'annual_income', 'loan_amount_lakh', 'credit_score', 'existing_loans', 'years_at_job'])
    ]
)


In [120]:
categorical_cols = ["employment_type", "city_tier"]

numeric_cols = [
    "age",
    "annual_income",
    "loan_amount_lakh",
    "credit_score",
    "existing_loans",
    "years_at_job"
]
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)


In [121]:
# Q1.5 (Fit) Call fit_transform on X_train with your ColumnTransformer, but only transform (not fit_transform) on X_test — fitting on the test set is a common mistake that causes a feature-count mismatch later. Then create a KNeighborsClassifier with n_neighbors=5 and fit it on the transformed training data.


preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)
X_train_transfer = preprocessor.fit_transform(x_train);
x_test_transfer = preprocessor.fit(x_test);

from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_transfer, y_train)



ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [122]:
# Q1.6 (Predict) Use the trained model to predict on the transformed test data.

pre = model.predict(x_test_transfer)


NotFittedError: This KNeighborsClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.